# MLE and Negative Log-Likelihood

**Goal:** Derive closed-form Gaussian MLE, implement NLL from scratch and validate against `torch.distributions.Normal`, then fit parameters by gradient descent and show convergence to the closed-form solution.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Generate Gaussian Data

Draw N i.i.d. samples from a known distribution N(μ_true, σ_true²). The goal of MLE is to recover these parameters from the data alone.

In [2]:
torch.manual_seed(42)

MU_TRUE = torch.tensor(4.0, device=device)
SIGMA_TRUE = torch.tensor(2.0, device=device)
N = 10_000

data = torch.randn(N, device=device) * SIGMA_TRUE + MU_TRUE
print(f"Data shape: {data.shape}, mean: {data.mean().item():.4f}, std: {data.std().item():.4f}")

Data shape: torch.Size([10000]), mean: 4.0096, std: 1.9971


## Closed-Form Gaussian MLE

The log-likelihood for i.i.d. Gaussian data is:

```
ℓ(μ, σ²) = -n/2 * log(2π σ²) - 1/(2σ²) * Σ(xᵢ - μ)²
```

Setting the derivative to zero gives the closed-form solutions:

```
μ̂_MLE  = (1/n) Σ xᵢ  = sample mean
σ̂²_MLE = (1/n) Σ (xᵢ - μ̂)²  = biased sample variance
```

In [3]:
# Closed-form MLE estimates
mu_mle = data.mean()
sigma2_mle = ((data - mu_mle) ** 2).mean()  # biased (divide by N)
sigma_mle = sigma2_mle.sqrt()

print(f"True  μ = {MU_TRUE.item():.4f}, σ = {SIGMA_TRUE.item():.4f}")
print(f"MLE   μ̂ = {mu_mle.item():.4f}, σ̂ = {sigma_mle.item():.4f}")

True  μ = 4.0000, σ = 2.0000
MLE   μ̂ = 4.0096, σ̂ = 1.9970


## Negative Log-Likelihood from Scratch

The Gaussian NLL is:

```
NLL(μ, σ; X) = -Σᵢ log p(xᵢ | μ, σ)
             = n/2 * log(2π) + n*log(σ) + 1/(2σ²) * Σ(xᵢ - μ)²
```

In [4]:
import math


def gaussian_nll_scratch(
    x: torch.Tensor, mu: torch.Tensor, sigma: torch.Tensor
) -> torch.Tensor:
    """Gaussian NLL computed from the formula (sum over samples)."""
    n = x.numel()
    return (
        0.5 * n * math.log(2.0 * math.pi)
        + n * torch.log(sigma)
        + 0.5 / sigma**2 * ((x - mu) ** 2).sum()
    )


nll_scratch = gaussian_nll_scratch(data, MU_TRUE, SIGMA_TRUE)

# Reference: sum of -Normal.log_prob
nll_torch = -torch.distributions.Normal(MU_TRUE, SIGMA_TRUE).log_prob(data).sum()

print(f"NLL from scratch: {nll_scratch.item():.4f}")
print(f"NLL from torch:   {nll_torch.item():.4f}")

assert torch.allclose(nll_scratch, nll_torch, atol=1e-2), (
    f"NLL mismatch: {nll_scratch.item()} vs {nll_torch.item()}"
)
print("NLL from scratch matches -Normal.log_prob.sum() ✓")

NLL from scratch: 21105.8789
NLL from torch:   21105.8789
NLL from scratch matches -Normal.log_prob.sum() ✓


## Fitting Parameters by Gradient Descent

Instead of the closed-form solution, we minimise the NLL with gradient descent. We use `torch.exp` to keep σ positive throughout optimisation.

In [5]:
torch.manual_seed(0)

# Parameterise sigma via log(sigma) to ensure positivity
mu_opt = torch.nn.Parameter(torch.tensor(0.0, device=device))
log_sigma_opt = torch.nn.Parameter(torch.tensor(0.0, device=device))

optimizer = torch.optim.Adam([mu_opt, log_sigma_opt], lr=0.05)

nll_history: list[float] = []
N_STEPS = 400

for step in range(N_STEPS):
    sigma_pos = torch.exp(log_sigma_opt)
    loss = gaussian_nll_scratch(data, mu_opt, sigma_pos)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    nll_history.append(loss.item())

mu_fitted = mu_opt.item()
sigma_fitted = torch.exp(log_sigma_opt).item()
print(f"After {N_STEPS} steps:  μ̂_gd = {mu_fitted:.5f},  σ̂_gd = {sigma_fitted:.5f}")
print(f"Closed-form MLE:      μ̂_cf = {mu_mle.item():.5f},  σ̂_cf = {sigma_mle.item():.5f}")

assert abs(mu_fitted - mu_mle.item()) < 0.01, "mu_gd did not converge to MLE!"
assert abs(sigma_fitted - sigma_mle.item()) < 0.01, "sigma_gd did not converge to MLE!"
print("Gradient descent converged to closed-form MLE ✓")

After 400 steps:  μ̂_gd = 4.00864,  σ̂_gd = 1.99699
Closed-form MLE:      μ̂_cf = 4.00955,  σ̂_cf = 1.99698
Gradient descent converged to closed-form MLE ✓


In [6]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(nll_history, lw=1.5)
ax.axhline(
    gaussian_nll_scratch(data, mu_mle, sigma_mle).item(),
    color="red", ls="--", label="NLL at closed-form MLE"
)
ax.set_xlabel("Gradient step")
ax.set_ylabel("NLL")
ax.set_title("NLL minimisation converges to closed-form MLE")
ax.legend()
plt.tight_layout()
plt.show()

/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_12028/2289705815.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## NLL → Cross-Entropy for Classification

For a categorical model with predicted probabilities **p** and one-hot label y:

```
NLL = -Σᵢ yᵢ · log pᵢ  =  cross-entropy(y, p)
```

Minimising NLL of the Bernoulli/Categorical distribution **is** minimising cross-entropy. `torch.nn.CrossEntropyLoss` computes `−log_softmax` internally, which equals `−Categorical.log_prob` (the NLL of the categorical distribution).

In [7]:
# One example: true label = class 2, logits from a small classifier
logits = torch.tensor([[1.2, 0.5, 2.1]], device=device)  # shape (1, 3)
target = torch.tensor([2], device=device)  # class 2

# Manual NLL = -log softmax at the correct class
log_softmax = torch.log_softmax(logits, dim=-1)
nll_manual = -log_softmax[0, target[0]].item()

# torch cross-entropy
nll_ce = torch.nn.CrossEntropyLoss()(logits, target).item()

print(f"Manual NLL (−log_softmax at target):  {nll_manual:.6f}")
print(f"nn.CrossEntropyLoss:                  {nll_ce:.6f}")
assert abs(nll_manual - nll_ce) < 1e-5
print("NLL == cross-entropy for categorical model ✓")

Manual NLL (−log_softmax at target):  0.475281
nn.CrossEntropyLoss:                  0.475281
NLL == cross-entropy for categorical model ✓


## Takeaways

- **MLE principle:** choose parameters that maximise p(data | θ). For Gaussians, this gives closed-form sample mean and biased variance.
- **NLL from scratch** matches `-Normal.log_prob.sum()` to floating-point precision.
- **Gradient descent on NLL** converges to the same solution as the closed form — confirming that neural network training is just maximum likelihood in disguise.
- **NLL = cross-entropy** for discrete models: minimising training loss is maximising the likelihood of the observed labels under the model's categorical distribution.